# 03 — Mitigasi bias (RQ2) dan penjelasan SHAP per kelompok (RQ3)

Model dasar: **regresi logistik** untuk kedua outcome (terbaik untuk diabetes; setara dengan model lain untuk disglikemia, dan paling mudah dijelaskan).

Empat strategi dibandingkan dengan lipatan nested CV yang sama seperti notebook 02:

| Kode | Strategi | Tahap pipeline |
| --- | --- | --- |
| Baseline | Regresi logistik + rekalibrasi + satu threshold (sensitivitas 80%) | — |
| Reweighing | Bobot latih per (kelompok × outcome) agar kelompok dan outcome independen (Kamiran & Calders) | Pra-proses |
| ExpGrad | Exponentiated Gradient dengan batasan *Equalized Odds* (fairlearn) | Saat pelatihan |
| Threshold per kelompok | Threshold berbeda per kelompok agar sensitivitas tiap kelompok ≈ 80% | Pasca-proses |

Mitigasi dijalankan untuk dua atribut sensitif secara terpisah: **ras/etnis** dan **kelompok usia**.
Ukuran keberhasilan: kesenjangan FNR dan Equalized Odds turun, sementara sensitivitas keseluruhan tetap ≈ 80%; biayanya dilihat dari penurunan spesifisitas.

Set `CEPAT = True` untuk uji kilat. Hasil final pakai `CEPAT = False`.

In [ ]:
# %pip install fairlearn

In [ ]:
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent  # repository root (run notebooks from notebooks/)
DATA = ROOT / "data" / "processed" / "nhanes_dm_belum_terdiagnosis.csv"
HASIL = ROOT / "hasil"
GAMBAR = HASIL / "gambar"
GAMBAR.mkdir(parents=True, exist_ok=True)

CEPAT = False
RANDOM_STATE = 42
N_LIPATAN = 5
N_ITER_TUNING = 4 if CEPAT else 15
N_BOOT = 100 if CEPAT else 500
TARGET_SENSITIVITAS = 0.80
MIN_KASUS = 30

try:
    from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
    ADA_FAIRLEARN = True
except ImportError:
    ADA_FAIRLEARN = False
    print("fairlearn belum terpasang: strategi ExpGrad dilewati. Jalankan sel %pip di atas.")
print("fairlearn:", ADA_FAIRLEARN, "| mode cepat:", CEPAT)

## 1. Data (sama dengan notebook 02)

In [ ]:
d = pd.read_csv(DATA)
d.loc[d["DBP"] < 30, "DBP"] = np.nan
d.loc[d["SBP"] < 60, "SBP"] = np.nan
d["PEREMPUAN"] = (d["JENIS_KELAMIN"] == "Perempuan").astype(int)
d["Y_DISGLIKEMIA"] = (d["HBA1C"] >= 5.7).astype(int)

OUTCOME = {
    "DM": ("DM_BELUM_TERDIAGNOSIS", "Diabetes belum terdiagnosis (HbA1c ≥ 6,5%)"),
    "DISGLIKEMIA": ("Y_DISGLIKEMIA", "Disglikemia belum terdiagnosis (HbA1c ≥ 5,7%)"),
}
PREDIKTOR = ["USIA", "PEREMPUAN", "BMI", "LINGKAR_PINGGANG", "SBP", "DBP", "RIWAYAT_HIPERTENSI",
             "RIWAYAT_KELUARGA_DM", "PERNAH_MEROKOK", "AKTIF_FISIK", "PENDIDIKAN"]
NAMA_FITUR = {"USIA": "Usia", "PEREMPUAN": "Perempuan", "BMI": "BMI", "LINGKAR_PINGGANG": "Lingkar pinggang",
              "SBP": "Tekanan sistolik", "DBP": "Tekanan diastolik", "RIWAYAT_HIPERTENSI": "Riwayat hipertensi",
              "RIWAYAT_KELUARGA_DM": "Riwayat keluarga DM", "PERNAH_MEROKOK": "Pernah merokok",
              "AKTIF_FISIK": "Aktif fisik", "PENDIDIKAN": "Pendidikan"}
ATRIBUT_MITIGASI = {"RAS_ETNIS": "Ras/etnis", "KEL_USIA": "Usia"}
STRATEGI = ["Baseline", "Reweighing"] + (["ExpGrad"] if ADA_FAIRLEARN else []) + ["Threshold per kelompok"]

X = d[PREDIKTOR].astype(float).values
W = d["WT_GABUNGAN"].values
print(f"N = {len(d):,} | strategi: {STRATEGI}")

## 2. Fungsi bantu
Sama dengan notebook 02: rekalibrasi Platt berbobot survei, threshold dengan sensitivitas berbobot ≥ 80%, metrik berbobot, dan *survey bootstrap* Rao–Wu.

In [ ]:
def logit(q):
    q = np.clip(q, 1e-6, 1 - 1e-6)
    return np.log(q / (1 - q))


def pilih_threshold(y, p, w, target=TARGET_SENSITIVITAS):
    pos = y == 1
    urut = np.argsort(-p[pos])
    p_pos, w_pos = p[pos][urut], w[pos][urut]
    kum = np.cumsum(w_pos) / w_pos.sum()
    return p_pos[min(np.searchsorted(kum, target), len(p_pos) - 1)]


def platt(skor_latih, y, w):
    m = LogisticRegression(C=1e6, max_iter=1000).fit(logit(skor_latih).reshape(-1, 1), y, sample_weight=w / w.mean())
    return lambda q: m.predict_proba(logit(q).reshape(-1, 1))[:, 1]


def bobot_reweighing(y, a):
    # w(a, y) = P(a) P(y) / P(a, y)
    df = pd.DataFrame({"a": a, "y": y})
    pa = df["a"].value_counts(normalize=True)
    py = df["y"].value_counts(normalize=True)
    pay = df.groupby(["a", "y"]).size() / len(df)
    return np.array([pa[ai] * py[yi] / pay[(ai, yi)] for ai, yi in zip(a, y)])


def metrik(y, p, yhat, w):
    nama = ["AUROC", "AUPRC", "Sensitivitas", "Spesifisitas", "PPV", "FNR", "FPR", "Brier", "O/E"]
    if w.sum() == 0:
        return {k: np.nan for k in nama}
    pos, neg = y == 1, y == 0
    dua = w[pos].sum() > 0 and w[neg].sum() > 0
    tp = w[pos & (yhat == 1)].sum(); fn = w[pos & (yhat == 0)].sum()
    tn = w[neg & (yhat == 0)].sum(); fp = w[neg & (yhat == 1)].sum()
    sens = tp / (tp + fn) if tp + fn > 0 else np.nan
    spes = tn / (tn + fp) if tn + fp > 0 else np.nan
    return {"AUROC": roc_auc_score(y, p, sample_weight=w) if dua else np.nan,
            "AUPRC": average_precision_score(y, p, sample_weight=w) if dua else np.nan,
            "Sensitivitas": sens, "Spesifisitas": spes,
            "PPV": tp / (tp + fp) if tp + fp > 0 else np.nan,
            "FNR": 1 - sens, "FPR": 1 - spes,
            "Brier": np.average((p - y) ** 2, weights=w),
            "O/E": np.average(y, weights=w) / np.average(p, weights=w)}


def bobot_replikasi(n_boot, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    psu = d["SDMVSTRA"].astype(int).astype(str) + "_" + d["SDMVPSU"].astype(int).astype(str)
    faktor = np.zeros((len(d), n_boot), dtype=np.float32)
    for _, g in d.groupby("SDMVSTRA"):
        daftar = psu[g.index].unique()
        n_h = len(daftar)
        if n_h < 2:
            faktor[g.index, :] = 1.0
            continue
        for b in range(n_boot):
            hitung = pd.Series(rng.choice(daftar, size=n_h - 1, replace=True)).value_counts()
            faktor[g.index, b] = psu[g.index].map(hitung).fillna(0).values * n_h / (n_h - 1)
    return faktor


FAKTOR = bobot_replikasi(N_BOOT)
print("Bobot replikasi:", FAKTOR.shape)

## 3. Nested CV untuk keempat strategi
Untuk setiap outcome × atribut sensitif × lipatan luar:
1. Imputasi median + standardisasi (dipasang di data latih saja).
2. Tuning `C` regresi logistik (3 lipatan dalam, skor AUPRC).
3. Skor *out-of-fold* data latih untuk setiap strategi → rekalibrasi Platt → threshold.
4. Prediksi untuk lipatan uji.

Bagian ini paling lama (ExpGrad melatih banyak model kecil).

In [ ]:
def latih_lr(C, Xa, ya, sw=None):
    return LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(Xa, ya, sample_weight=sw)


def latih_strategi(nama, C, Xa, ya, aa):
    if nama == "Reweighing":
        m = latih_lr(C, Xa, ya, bobot_reweighing(ya, aa))
        return lambda Xb: m.predict_proba(Xb)[:, 1]
    if nama == "ExpGrad":
        eg = ExponentiatedGradient(LogisticRegression(C=C, class_weight="balanced", max_iter=5000),
                                   constraints=EqualizedOdds(), eps=0.01)
        eg.fit(Xa, ya, sensitive_features=aa)
        return lambda Xb: eg._pmf_predict(Xb)[:, 1]
    m = latih_lr(C, Xa, ya)  # Baseline (juga dasar Threshold per kelompok)
    return lambda Xb: m.predict_proba(Xb)[:, 1]


def skor_oof(nama, C, Xa, ya, aa, wa):
    s = np.zeros(len(ya))
    for a, b in StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE).split(Xa, ya):
        s[b] = latih_strategi(nama, C, Xa[a], ya[a], aa[a])(Xa[b])
    return s


def threshold_per_kelompok(y, p, w, a):
    thr_global = pilih_threshold(y, p, w)
    thr = {}
    for g in np.unique(a):
        m = a == g
        thr[g] = pilih_threshold(y[m], p[m], w[m]) if y[m].sum() >= MIN_KASUS * (N_LIPATAN - 1) / N_LIPATAN else thr_global
    return thr, thr_global


def jalankan(y, A):
    strat = d["SIKLUS"].astype(str) + "_" + y.astype(str)
    luar = StratifiedKFold(n_splits=N_LIPATAN, shuffle=True, random_state=RANDOM_STATE)
    prob = {s: np.zeros(len(y)) for s in STRATEGI}
    pred = {s: np.zeros(len(y), dtype=int) for s in STRATEGI}
    nilai_C = []
    for tr, te in luar.split(X, strat):
        prep = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]).fit(X[tr])
        Xtr, Xte = prep.transform(X[tr]), prep.transform(X[te])
        cari = RandomizedSearchCV(LogisticRegression(class_weight="balanced", max_iter=5000),
                                  {"C": loguniform(1e-3, 10)}, n_iter=N_ITER_TUNING, scoring="average_precision",
                                  cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                  random_state=RANDOM_STATE).fit(Xtr, y[tr])
        C = cari.best_params_["C"]
        nilai_C.append(C)
        for s in STRATEGI:
            dasar = "Baseline" if s == "Threshold per kelompok" else s
            oof = skor_oof(dasar, C, Xtr, y[tr], A[tr], W[tr])
            kal = platt(oof, y[tr], W[tr])
            p_te = kal(latih_strategi(dasar, C, Xtr, y[tr], A[tr])(Xte))
            prob[s][te] = p_te
            if s == "Threshold per kelompok":
                thr, thr_global = threshold_per_kelompok(y[tr], kal(oof), W[tr], A[tr])
                pred[s][te] = (p_te >= np.array([thr.get(g, thr_global) for g in A[te]])).astype(int)
            else:
                pred[s][te] = (p_te >= pilih_threshold(y[tr], kal(oof), W[tr])).astype(int)
    return prob, pred, nilai_C


HASIL_CV = {}
for kode, (kol, ket) in OUTCOME.items():
    y = d[kol].values.astype(int)
    for atr, label in ATRIBUT_MITIGASI.items():
        mulai = time.time()
        A = d[atr].astype(str).values
        HASIL_CV[(kode, atr)] = jalankan(y, A)
        print(f"{ket} | atribut {label}: selesai ({(time.time() - mulai) / 60:.1f} menit)")

## 4. Performa dan kesenjangan per strategi
Kesenjangan dihitung hanya antar kelompok dengan ≥ 30 kasus. CI 95% dari survey bootstrap.

In [ ]:
def ringkas(y, p, yhat, A):
    kel = [g for g in np.unique(A) if y[A == g].sum() >= MIN_KASUS]

    def satu(w):
        total = metrik(y, p, yhat, w)
        per = {g: metrik(y[A == g], p[A == g], yhat[A == g], w[A == g]) for g in kel}
        fnr = [per[g]["FNR"] for g in kel]; fpr = [per[g]["FPR"] for g in kel]
        total["Selisih FNR"] = max(fnr) - min(fnr)
        total["EOD"] = max(max(fnr) - min(fnr), max(fpr) - min(fpr))
        return total, per

    titik, per = satu(W)
    rep = pd.DataFrame([satu(W * FAKTOR[:, b])[0] for b in range(FAKTOR.shape[1])])
    baris = {}
    for k, v in titik.items():
        baris[k] = v
        baris[k + "_lo"] = np.nanpercentile(rep[k], 2.5)
        baris[k + "_hi"] = np.nanpercentile(rep[k], 97.5)
    return baris, per


ringkasan, per_kelompok = [], []
for (kode, atr), (prob, pred, _) in HASIL_CV.items():
    y = d[OUTCOME[kode][0]].values.astype(int)
    A = d[atr].astype(str).values
    for s in STRATEGI:
        baris, per = ringkas(y, prob[s], pred[s], A)
        ringkasan.append({"Outcome": kode, "Atribut": ATRIBUT_MITIGASI[atr], "Strategi": s, **baris})
        for g, m in per.items():
            per_kelompok.append({"Outcome": kode, "Atribut": ATRIBUT_MITIGASI[atr], "Strategi": s, "Kelompok": g,
                                 "Kasus": int(y[A == g].sum()), **m})

ringkasan = pd.DataFrame(ringkasan)
per_kelompok = pd.DataFrame(per_kelompok)
ringkasan.to_csv(HASIL / "03_mitigasi_ringkasan.csv", index=False)
per_kelompok.to_csv(HASIL / "03_mitigasi_per_kelompok.csv", index=False)

def fmt(r, k, n=3):
    return f"{r[k]:.{n}f} ({r[k + '_lo']:.{n}f}–{r[k + '_hi']:.{n}f})"

ringkasan.apply(lambda r: pd.Series({
    "Outcome": r["Outcome"], "Atribut": r["Atribut"], "Strategi": r["Strategi"],
    "AUROC": fmt(r, "AUROC"), "Sensitivitas": fmt(r, "Sensitivitas"), "Spesifisitas": fmt(r, "Spesifisitas"),
    "Selisih FNR": fmt(r, "Selisih FNR"), "EOD": fmt(r, "EOD")}), axis=1)

In [ ]:
# FNR per kelompok sebelum dan sesudah mitigasi
per_kelompok.pivot_table(index=["Outcome", "Atribut", "Kelompok"], columns="Strategi", values="FNR").round(3)

## 5. Gambar trade-off: keadilan vs biaya
Sumbu x = selisih FNR antar kelompok (makin kiri makin adil). Sumbu y = spesifisitas keseluruhan (makin tinggi makin sedikit positif palsu). Sensitivitas keseluruhan dijaga ≈ 80% di semua strategi.

In [ ]:
WARNA = {"Baseline": "#52514e", "Reweighing": "#2a78d6", "ExpGrad": "#eb6834", "Threshold per kelompok": "#1baf7a"}
SIMBOL = {"Baseline": "o", "Reweighing": "s", "ExpGrad": "^", "Threshold per kelompok": "D"}
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#8a8984", "xtick.color": "#52514e", "ytick.color": "#52514e"})

panel = [(k, a) for k in OUTCOME for a in ATRIBUT_MITIGASI]
fig, axes = plt.subplots(1, len(panel), figsize=(3.3 * len(panel), 3.6), sharey=False)
for ax, (kode, atr) in zip(axes, panel):
    sub = ringkasan[(ringkasan.Outcome == kode) & (ringkasan.Atribut == ATRIBUT_MITIGASI[atr])]
    for _, r in sub.iterrows():
        c = WARNA[r["Strategi"]]
        ax.errorbar(r["Selisih FNR"], r["Spesifisitas"],
                    xerr=[[r["Selisih FNR"] - r["Selisih FNR_lo"]], [r["Selisih FNR_hi"] - r["Selisih FNR"]]],
                    yerr=[[r["Spesifisitas"] - r["Spesifisitas_lo"]], [r["Spesifisitas_hi"] - r["Spesifisitas"]]],
                    fmt=SIMBOL[r["Strategi"]], color=c, ms=7, mec="white", mew=1, elinewidth=1.2, capsize=0,
                    label=r["Strategi"])
    ax.set_title(f"{'Diabetes' if kode == 'DM' else 'Disglikemia'} — {ATRIBUT_MITIGASI[atr]}", fontsize=9, loc="left")
    ax.set_xlabel("Selisih FNR antar kelompok")
    ax.grid(color="#e6e5e0", lw=0.6)
    ax.set_xlim(left=0)
axes[0].set_ylabel("Spesifisitas keseluruhan")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=len(STRATEGI), frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig(GAMBAR / "03A_tradeoff_mitigasi.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. SHAP per kelompok (RQ3)
Untuk regresi logistik, nilai SHAP dapat dihitung tepat: kontribusi fitur *j* = koefisien × (nilai terstandar − rata-rata), pada skala log-odds (setara `shap.LinearExplainer`).
Kita bandingkan rata-rata kontribusi setiap fitur di antara **kasus diabetes** pada kelompok Asia dan kulit putih, lalu di antara kasus Asia yang **terlewat** vs yang **terdeteksi**.

In [ ]:
y = d["DM_BELUM_TERDIAGNOSIS"].values.astype(int)
prep = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]).fit(X)
Xs = prep.transform(X)
C_median = float(np.median(HASIL_CV[("DM", "RAS_ETNIS")][2]))
lr = LogisticRegression(C=C_median, class_weight="balanced", max_iter=5000).fit(Xs, y)
SHAP = lr.coef_[0] * (Xs - Xs.mean(axis=0))
shap_df = pd.DataFrame(SHAP, columns=[NAMA_FITUR[c] for c in PREDIKTOR])

pred_base = HASIL_CV[("DM", "RAS_ETNIS")][1]["Baseline"]
ras = d["RAS_ETNIS"].values
kasus = y == 1

tabel_shap = pd.DataFrame({
    "Kasus NH Asian": shap_df[kasus & (ras == "NH Asian")].mean(),
    "Kasus NH White": shap_df[kasus & (ras == "NH White")].mean(),
    "Asia terlewat (FN)": shap_df[kasus & (ras == "NH Asian") & (pred_base == 0)].mean(),
    "Asia terdeteksi (TP)": shap_df[kasus & (ras == "NH Asian") & (pred_base == 1)].mean(),
})
tabel_shap["Selisih Asia − Putih"] = tabel_shap["Kasus NH Asian"] - tabel_shap["Kasus NH White"]
tabel_shap = tabel_shap.sort_values("Selisih Asia − Putih")
tabel_shap.to_csv(HASIL / "03_shap_per_kelompok.csv")

print("Rata-rata nilai klinis di antara kasus diabetes:")
display(d[kasus & d["RAS_ETNIS"].isin(["NH Asian", "NH White"])]
        .groupby("RAS_ETNIS")[["BMI", "LINGKAR_PINGGANG", "USIA", "SBP"]].mean().round(1))
tabel_shap.round(3)

In [ ]:
fitur = list(tabel_shap.index)
ypos = np.arange(len(fitur))
fig, ax = plt.subplots(figsize=(6.4, 0.38 * len(fitur) + 1.2))
for kol, warna, mk, geser in [("Kasus NH Asian", "#2a78d6", "o", 0.12), ("Kasus NH White", "#eb6834", "s", -0.12)]:
    ax.plot(tabel_shap[kol], ypos + geser, mk, color=warna, ms=6, mec="white", mew=1, label=kol.replace("Kasus ", ""))
for i, f in enumerate(fitur):
    ax.plot([tabel_shap.loc[f, "Kasus NH Asian"], tabel_shap.loc[f, "Kasus NH White"]], [i + 0.12, i - 0.12],
            color="#c3c2b7", lw=1, zorder=0)
ax.axvline(0, color="#52514e", lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(fitur)
ax.set_xlabel("Rata-rata kontribusi SHAP pada kasus diabetes (log-odds)")
ax.set_title("Kontribusi fitur pada kasus diabetes belum terdiagnosis:\nAsia vs kulit putih (regresi logistik)", fontsize=9, loc="left")
ax.grid(axis="x", color="#e6e5e0", lw=0.6)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(GAMBAR / "03B_shap_asia_vs_putih.png", dpi=300)
plt.show()

## Hasil yang tersimpan
| File | Isi |
| --- | --- |
| `hasil\03_mitigasi_ringkasan.csv` | Performa dan kesenjangan per strategi × outcome × atribut, dengan CI 95% |
| `hasil\03_mitigasi_per_kelompok.csv` | Metrik per kelompok untuk setiap strategi |
| `hasil\03_shap_per_kelompok.csv` | Rata-rata SHAP per fitur: Asia vs putih, terlewat vs terdeteksi |
| `hasil\gambar\03A_tradeoff_mitigasi.png`, `03B_shap_asia_vs_putih.png` | Gambar naskah (300 dpi) |

Setelah ini: menulis bagian Methods dan Results naskah.